In [ ]:
# Import libraries
import sqlite3
import pandas as pd
import json

In [ ]:
def execute_query(db_file, query):
    """
    Executes a SQL SELECT query and returns column names and data.
    """
    try:
        with sqlite3.connect(db_file) as conn:
            cursor = conn.cursor()
            cursor.execute(query)
            data = cursor.fetchall()
            column_names = [description[0] for description in cursor.description]
            return column_names, data
    except sqlite3.Error as e:
        print(f"Error executing query: {e}")
        return [], []

In [ ]:
# SQL query to filter papers based on area mentions
query = """
    WITH Places(Place) AS (
        VALUES 
            ('GALVESTON'), ('HOUSTON'), ('LAKE%JACKSON'), ('LEAGUE%CITY'), ('TEXAS%CITY'),
            ('TIKI%ISLAND'), ('ANAHUAC'), ('BAYTOWN'), ('JAMAICA%BEACH'), ('LA%PORTE'),
            ('CHANNELVIEW'), ('HARRIS%COUNTY'), ('HURRICANE%HARVEY'), ('MONT%BELVIEU'),
            ('SEABROOK'), ('SAN%JACINTO%RIVER'), ('TRINITY%RIVER'), ('ATASCOCITA'),
            ('PEARLAND'), ('SUGAR%LAND'), ('DEER%PARK')
    )
    SELECT DISTINCT pu.IDPaper, pu.DI AS DOI, pu.PY AS Year, pu.TI AS Title, pu.AF AS Author, 
                    pu.AB AS Abstract, pu.PU AS Publisher, pu.JI AS Journal, pu.Area, 
                    pu.C1 AS Author_Affiliations, pu.ID AS Keywords, pu.WC AS Categories
    FROM Places pl 
    LEFT JOIN TBLPapers pu ON pu.Area LIKE CONCAT('%', pl.Place, '%') 
    WHERE pu.IDPaper IS NOT NULL
    ORDER BY pu.IDPaper;
"""

# Execute query and load into DataFrame
column_names, data = execute_query("data.db", query)
df = pd.DataFrame(data, columns=column_names)

In [ ]:
def filter_out_irrelevant_papers(df, ids_to_remove):
    """
    Removes manually flagged records from the DataFrame.
    """
    return df[~df['IDPaper'].isin(ids_to_remove)]

# Manually identified false positives
papers_to_remove = [
    90179, 203066, 54983, 202878, 11930, 49479, 116368, 118853, 203193,
    15246, 42321, 49519, 82428, 104939, 112509, 114928, 148130
]

# Filter out non-relevant papers
df_filtered = filter_out_irrelevant_papers(df, papers_to_remove)


In [ ]:
# Extract metadata fields and structure as JSON
df_metadata = df_filtered[[
    'IDPaper', 'DOI', 'Year', 'Title', 'Author', 'Publisher', 'Journal',
    'Area', 'Author_Affiliations', 'Keywords', 'Categories'
]]

# Normalize semi-structured fields
df_metadata["Author"] = df_metadata["Author"].apply(lambda x: [a.strip() for a in x.split(";")] if pd.notna(x) else [])
df_metadata["Keywords"] = df_metadata["Keywords"].apply(lambda x: [k.strip() for k in x.split(";")] if pd.notna(x) else [])
df_metadata["Categories"] = df_metadata["Categories"].apply(lambda x: [c.strip() for c in x.split(";")] if pd.notna(x) else [])
df_metadata["Area"] = df_metadata["Area"].apply(lambda x: [a.strip() for a in x.split(";")] if pd.notna(x) else [])

df_metadata = df_metadata.rename(columns={"Area": "Study Area"})

# Save to JSON
metadata_dict = df_metadata.set_index("IDPaper").to_dict(orient="index")
with open("metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata_dict, f, indent=4, ensure_ascii=False)

print(f"Filtered metadata exported to metadata.json with {len(df_metadata)} records.")

In [ ]:
# Download instructions

print("\nDownload Required:")
print("Please manually download the PDFs corresponding to the selected papers and save them in the 'Houston_pdfs/' directory.")
print("Each file should be named after the IDPaper value, e.g., '753.pdf'.")
